# 01 — TFPT structured-residual prediction

Visualises the TFPT prediction for the achromatic, dyonic residual intercept around a compact object:

$$ \beta_{\mathrm{BH}}(r) = \frac{Q_e^{\mathrm{eff}}\,Q_m^{\mathrm{eff}}}{256\,\pi^4\,r^2} = 16\,c_3^4\,\frac{Q_e^{\mathrm{eff}}\,Q_m^{\mathrm{eff}}}{r^2}. $$

The coupling $1/(256\pi^4)$ is fixed by the same TFPT branch data that fixes $\alpha$ and the global cosmic-birefringence amplitude $\beta=0.2424^\circ$. Only the geometric weights $Q_e^{\mathrm{eff}}, Q_m^{\mathrm{eff}}$ and the emission radius are model dependent.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

from tfpt_eht import (
    BETA_DEG,
    BETA_RAD,
    BlackHoleGeometry,
    C3,
    DELTA_TOP,
    PHI0,
    TFPT_COUPLING,
    beta_bh_profile,
    chi0_tfpt,
    expected_amplitude_at,
)

print(f'phi0          = {PHI0:.12e}')
print(f'c3            = {C3:.12e}')
print(f'beta          = {BETA_DEG:.6f} deg  ({BETA_RAD:.4e} rad)')
print(f'delta_top     = {DELTA_TOP:.4e}')
print(f'TFPT coupling = {TFPT_COUPLING:.4e}   (1/(256 pi^4) = 16 c3^4 = delta_top/3)')
expected_amplitude_at(10.0)

## Radial profile (unit weights)

The amplitude at $r = 10\,r_g$ with $Q_e^{\mathrm{eff}} = Q_m^{\mathrm{eff}} = 1$ is $\beta_{\mathrm{BH}} \approx 4 \times 10^{-7}$ rad $\approx 23\,\mu\mathrm{deg}$.

Realistic dyonic regions in GRMHD simulations of M87\* support $Q_e^{\mathrm{eff}} Q_m^{\mathrm{eff}}$ between $1$ and $10^3$; the prediction scales linearly with this product.

In [ ]:
radii = np.logspace(np.log10(4.0), np.log10(40.0), 200)
fig, ax = plt.subplots(figsize=(6.0, 4.5))
for q in [1.0, 10.0, 100.0]:
    ax.loglog(radii, beta_bh_profile(radii, q_e_eff=q, q_m_eff=q) * 180 / math.pi * 1e6,
              label=f'Q_e Q_m = {int(q*q)}')
ax.set_xlabel('radius  [r_g]')
ax.set_ylabel(r'$|\beta_{\mathrm{BH}}(r)|$  [\u00b5deg]')
ax.set_title('TFPT structured residual amplitude')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 2-D map of $\chi_0^{\mathrm{TFPT}}(x)$

The signed contribution to the image-plane polarization-angle intercept, restricted to the EHT annulus $r_{\mathrm{inner}} \le r \le r_{\mathrm{outer}}$.

In [ ]:
side = np.linspace(-30.0, 30.0, 256)
x, y = np.meshgrid(side, side, indexing='xy')
geom = BlackHoleGeometry(r_inner=4.0, r_outer=25.0, sign_orientation=+1)
chi = chi0_tfpt((x, y), geom, q_e_eff=10.0, q_m_eff=10.0)

fig, ax = plt.subplots(figsize=(5.5, 5.0))
im = ax.imshow(chi * 1e6, origin='lower', extent=(-30, 30, -30, 30),
               cmap='RdBu_r', interpolation='nearest')
ax.set_xlabel('x  [r_g]')
ax.set_ylabel('y  [r_g]')
ax.set_title(r'$\chi_0^{\mathrm{TFPT}}(x)$,  $Q_e Q_m = 100$  [\u00b5rad]')
fig.colorbar(im, ax=ax, shrink=0.85)
plt.tight_layout()
plt.show()

## Sign flip under $E\!\cdot\!B$ reversal

The third null test: reversing the effective $E\!\cdot\!B$ orientation flips the sign of the entire map.

In [ ]:
geom_plus = BlackHoleGeometry(r_inner=4.0, r_outer=25.0, sign_orientation=+1)
geom_minus = BlackHoleGeometry(r_inner=4.0, r_outer=25.0, sign_orientation=-1)
chi_plus = chi0_tfpt((x, y), geom_plus, q_e_eff=10.0, q_m_eff=10.0)
chi_minus = chi0_tfpt((x, y), geom_minus, q_e_eff=10.0, q_m_eff=10.0)

fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.5))
vlim = np.max(np.abs(chi_plus)) * 1e6
for ax, data, title in zip(axes, [chi_plus, chi_minus], ['+ orientation', '\u2212 orientation']):
    im = ax.imshow(data * 1e6, origin='lower', extent=(-30, 30, -30, 30),
                    cmap='RdBu_r', vmin=-vlim, vmax=+vlim, interpolation='nearest')
    ax.set_xlabel('x  [r_g]')
    ax.set_ylabel('y  [r_g]')
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.85)
fig.suptitle(r'TFPT prediction: $\chi_0$ flips under $E\!\cdot\!B$ reversal')
plt.tight_layout()
plt.show()